# 02a · Protocol A - 50 training rounds + **5 recovery rounds**
Produces the raw data for **Table II** (ACC/ASR at the end of 5 recovery rounds), **Fig. 6** (ACC/ASR/MIA curves),
and it is also the source of the λ_OT=0.1 point of Table IV and Fig. 10.
Paper settings: 4 datasets × 6 methods × seeds {0, 1}.  Local optimizer: Adam (kept from the repo on purpose).

Resumable: finished (dataset, method, seed) results in `results/raw/` are skipped, so you can stop and re-run.
The 49 shared training rounds are trained **once per (dataset, seed)** and cached in `artifacts/checkpoints/`;
every method then starts from the same pre-unlearning model (identical to what the repo's scripts recompute each time).

Tip: validate the pipeline first with `DATASETS=["CIFAR10"]`, `SEEDS=(0,)`.
**Colab sessions:** the VM is wiped when the runtime is recycled and free sessions are time-limited. Every finished run is saved at once, so just re-run the cell after a disconnect;
call `colab.zip_results()` (last cell) regularly to download your results.

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        break
print("project root:", ROOT)

In [ ]:
from fuguard_repro.config import IMAGE_DATASETS, METHODS, PROTOCOLS
from fuguard_repro.runner import run_grid, standard_jobs

DATASETS = IMAGE_DATASETS
SELECTED_METHODS = METHODS
SEEDS = PROTOCOLS["table2"]["seeds"]                      # (0, 1)
RECOVERY_ROUNDS = PROTOCOLS["table2"]["recovery_rounds"]  # 5

In [ ]:
run_grid("table2", DATASETS, standard_jobs(SELECTED_METHODS), SEEDS, RECOVERY_ROUNDS)

In [ ]:
from fuguard_repro.aggregate import table2_numeric, table2_df
display(table2_df(table2_numeric()))

**What to check:** the FedAvg row should land near the paper's (SVHN 83.9 / CIFAR-10 64.9 / CIFAR-100 45.1 / EuroSAT 73.8 % ACC).
If it does not, fix the data/training side before looking at unlearning. Full comparison table → `03_analysis/03a_tables.ipynb`.